# PromptGFM-Bio — ARC Labs Workstation Notebook
**Gene-Phenotype Prediction for Rare Diseases**

| Spec | Value |
|---|---|
| CPU | Intel i9-14900K |
| RAM | 128 GB |
| GPU | NVIDIA GeForce RTX 4090 (24 GB VRAM) |
| CUDA | 13.0 · Driver 580.65.06 |
| Disk | 512 GB |

### How this notebook works
- **Single project root**: every cell uses `PROJECT_ROOT` — no path confusion.
- **VRAM-aware**: auto-detects free GPU memory and scales batch size accordingly.
- **Resumable**: data persists on disk between Jupyter sessions (5-day window).
- **Secrets**: GitHub token + W&B key loaded from `.env` in the project root.

> ⚠️ **5-day data retention** — back up to GitHub/HuggingFace before your window expires.

## 0. 🔒 Master Path Setup (RUN THIS FIRST — ALWAYS)

This cell sets `PROJECT_ROOT` and `os.chdir()` so **every cell below**
uses the same directory. Nothing else in this notebook defines paths independently.

In [1]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  EDIT THIS ONE LINE if your project is in a different location         ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, sys, subprocess
from pathlib import Path

GITHUB_URL   = "https://github.com/pes1ug23am910/PromptGFM-Bio.git"
PROJECT_ROOT = Path("/home/mluser/projects_yash/new_project/PromptGFM-Bio").resolve()

# ── Ensure directory exists ───────────────────────────────────────────────
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# ═══════════════════════════════════════════════════════════════════════════
# AUTO-CLONE: if scripts/ and configs/ are missing, pull the repo code
# Safe for non-empty directories — won't clobber .env, data/, hf_cache/
# ═══════════════════════════════════════════════════════════════════════════
if not (PROJECT_ROOT / "scripts").is_dir() or not (PROJECT_ROOT / "configs").is_dir():
    print("⚠️  scripts/ and/or configs/ not found — cloning repo code...")
    print()

    # Load .env early to get GITHUB_TOKEN for private repo
    _env = PROJECT_ROOT / ".env"
    if _env.is_file():
        for _line in _env.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _, _v = _line.partition("=")
                os.environ[_k.strip()] = _v.strip()

    _token = os.environ.get("GITHUB_TOKEN", "")
    _auth_url = GITHUB_URL.replace("https://", f"https://{_token}@") if _token else GITHUB_URL

    if (PROJECT_ROOT / ".git").is_dir():
        # Already a git repo but missing files — just pull
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=True)
    else:
        # Not a git repo — init and pull (safe for non-empty dirs)
        subprocess.run(["git", "init"], cwd=str(PROJECT_ROOT), check=True,
                       capture_output=True)
        subprocess.run(["git", "remote", "add", "origin", _auth_url],
                       cwd=str(PROJECT_ROOT), capture_output=True)

        result = subprocess.run(
            ["git", "fetch", "origin"],
            cwd=str(PROJECT_ROOT), capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"   ❌ git fetch failed: {result.stderr.strip()}")
            if not _token:
                print("   Repo is private — add GITHUB_TOKEN to .env")
            raise RuntimeError("Cannot fetch repo code")

        # Detect default branch
        _branch_result = subprocess.run(
            ["git", "remote", "show", "origin"],
            cwd=str(PROJECT_ROOT), capture_output=True, text=True
        )
        _default_branch = "main"
        for _bl in _branch_result.stdout.splitlines():
            if "HEAD branch" in _bl:
                _default_branch = _bl.split(":")[-1].strip()
                break

        subprocess.run(
            ["git", "checkout", "-f", f"origin/{_default_branch}", "-b", _default_branch],
            cwd=str(PROJECT_ROOT), capture_output=True
        )

    if (PROJECT_ROOT / "scripts").is_dir() and (PROJECT_ROOT / "configs").is_dir():
        print("   ✅ Repo code cloned successfully")
    else:
        print("   ❌ Clone finished but scripts/configs still missing!")
        raise RuntimeError("Repo doesn't contain expected scripts/ and configs/")
else:
    print("✅ Repo code already present (scripts/ and configs/ found)")

# ── Derived path variables ────────────────────────────────────────────────
SCRIPTS_DIR  = PROJECT_ROOT / "scripts"
CONFIGS_DIR  = PROJECT_ROOT / "configs"
DATA_DIR     = PROJECT_ROOT / "data"
CKPT_DIR     = PROJECT_ROOT / "checkpoints" / "promptgfm_film"
LOGS_DIR     = PROJECT_ROOT / "logs"
ENV_FILE     = PROJECT_ROOT / ".env"

# ── HF cache ─────────────────────────────────────────────────────────────
HF_CACHE_DIR = str(PROJECT_ROOT / "hf_cache")
os.environ["HF_HOME"]               = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"]     = HF_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE_DIR

# ── Resume flags ──────────────────────────────────────────────────────────
RESUME_HF_CACHE     = False
RESUME_DATA         = False
RESUME_CHECKPOINTS  = False

# ── Status report ─────────────────────────────────────────────────────────
print()
print(f"PROJECT_ROOT   = {PROJECT_ROOT}")
print(f"CWD            = {Path.cwd()}")
print(f"  scripts/     : {'✅' if SCRIPTS_DIR.is_dir() else '❌'}")
print(f"  configs/     : {'✅' if CONFIGS_DIR.is_dir() else '❌'}")
print(f"  data/        : {'✅' if DATA_DIR.is_dir() else '⬜ will create'}")
print(f"  .env         : {'✅' if ENV_FILE.is_file() else '❌ MISSING'}")
print(f"  .git/        : {'✅' if (PROJECT_ROOT / '.git').is_dir() else '❌'}")
print(f"  hf_cache/    : {'✅' if Path(HF_CACHE_DIR).is_dir() else '⬜ will create'}")
print(f"  RESUME_HF_CACHE    = {RESUME_HF_CACHE}")
print(f"  RESUME_DATA        = {RESUME_DATA}")
print(f"  RESUME_CHECKPOINTS = {RESUME_CHECKPOINTS}")

✅ Repo code already present (scripts/ and configs/ found)

PROJECT_ROOT   = /home/mluser/projects_yash/new_project/PromptGFM-Bio
CWD            = /home/mluser/projects_yash/new_project/PromptGFM-Bio
  scripts/     : ✅
  configs/     : ✅
  data/        : ✅
  .env         : ✅
  .git/        : ❌
  hf_cache/    : ✅
  RESUME_HF_CACHE    = False
  RESUME_DATA        = False
  RESUME_CHECKPOINTS = False


## 1. Load Secrets from `.env`

Your `.env` file should be at:
```
/home/mluser/projects_yash/new_project/PromptGFM-Bio/.env
```
Contents:
```
GITHUB_TOKEN=ghp_xxxxxxxxxxxxxxxxxxxx
WANDB_API_KEY=xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
```

In [2]:
import os

print(f"Loading: {ENV_FILE}")

if ENV_FILE.is_file():
    for line in ENV_FILE.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, val = line.partition("=")
            os.environ[key.strip()] = val.strip()
    print("✅ .env loaded")
else:
    print(f"❌ .env not found at {ENV_FILE}")
    print(f'   Create it:  echo -e "GITHUB_TOKEN=ghp_xxx\nWANDB_API_KEY=xxx" > {ENV_FILE}')

print(f"  GITHUB_TOKEN  : {'✅ set' if os.environ.get('GITHUB_TOKEN') else '❌ missing'}")
print(f"  WANDB_API_KEY : {'✅ set' if os.environ.get('WANDB_API_KEY') else '❌ missing'}")

Loading: /home/mluser/projects_yash/new_project/PromptGFM-Bio/.env
✅ .env loaded
  GITHUB_TOKEN  : ✅ set
  WANDB_API_KEY : ✅ set


## 2. GPU Diagnostics & VRAM Budget

In [3]:
import subprocess, os

subprocess.run(["nvidia-smi"])
print()

try:
    import torch
    if torch.cuda.is_available():
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total,memory.free",
             "--format=csv,nounits,noheader"],
            capture_output=True, text=True
        )
        smi_used, smi_total, smi_free = [int(x.strip()) for x in result.stdout.strip().split(",")]
        usable_mb = smi_free - 2048

        print(f"GPU           : {torch.cuda.get_device_name(0)}")
        print(f"Total VRAM    : {smi_total:,} MiB")
        print(f"Used (idle)   : {smi_used:,} MiB")
        print(f"Free VRAM     : {smi_free:,} MiB")
        print(f"Usable        : {usable_mb:,} MiB (after 2 GB safety margin)")
        print(f"PyTorch       : {torch.__version__}")
        print(f"CUDA          : {torch.version.cuda}")

        os.environ["_VRAM_FREE_MB"]   = str(smi_free)
        os.environ["_VRAM_USABLE_MB"] = str(usable_mb)
    else:
        print("⚠️  No GPU detected")
except ImportError:
    print("⚠️  PyTorch not installed yet — run Step 3 first")

Thu Apr  2 16:50:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.65.06              Driver Version: 580.65.06      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:01:00.0 Off |                  Off |
| 34%   39C    P8             17W /  450W |     117MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Install PyTorch & PyTorch Geometric

Installs into your existing venv. CUDA 12.4 wheels are compatible with your CUDA 13.0 driver.

In [4]:
import subprocess, sys

# ── PyTorch ───────────────────────────────────────────────────────────────
try:
    import torch
    print(f"PyTorch {torch.__version__} already installed (CUDA {torch.version.cuda})")
except ImportError:
    print("Installing PyTorch...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "torch", "torchvision", "torchaudio",
        "--index-url", "https://download.pytorch.org/whl/cu124"
    ], check=True)
    import torch
    print(f"✅ PyTorch {torch.__version__} installed")

# ── PyG ───────────────────────────────────────────────────────────────────
TORCH_VER = torch.__version__.split("+")[0]
CUDA_VER  = torch.version.cuda.replace(".", "")
WHEEL_URL = f"https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_VER}.html"
print(f"PyG wheel source: {WHEEL_URL}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "-f", WHEEL_URL,
     "torch-scatter", "torch-sparse", "torch-cluster",
     "torch-spline-conv", "torch-geometric"],
    check=True
)
print("✅ PyTorch Geometric installed")

PyTorch 2.6.0+cu124 already installed (CUDA 12.4)
PyG wheel source: https://data.pyg.org/whl/torch-2.6.0+cu124.html
✅ PyTorch Geometric installed


## 4. Install Extra Dependencies

In [5]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "--upgrade", "setuptools", "wheel", "pip"], check=True)

extra = [
    "transformers>=4.40.0",
    "sentence-transformers>=2.7.0",
    "biopython>=1.84",
    "networkx>=3.2",
    "wandb>=0.17.0",
    "python-dotenv>=1.0.0",
    "huggingface_hub",
    "pyyaml",
]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + extra, check=True)
print("✅ Extra packages installed")

✅ Extra packages installed


## 5. Git Pull Latest Code

Your repo is already cloned at `PROJECT_ROOT`. This just pulls the latest changes.

In [6]:
import subprocess, os

os.environ["GIT_TERMINAL_PROMPT"] = "0"

if (PROJECT_ROOT / ".git").is_dir():
    result = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("✅ Latest code pulled")
        if result.stdout.strip():
            print(f"   {result.stdout.strip()}")
    else:
        print("⚠️  git pull failed — continuing with existing code")
        print(f"   {result.stderr.strip()}")
else:
    print("⚠️  Not a git repo — skipping pull (code was set up in Cell 0)")

⚠️  Not a git repo — skipping pull (code was set up in Cell 0)


In [7]:
'''
import os, sys, subprocess
from pathlib import Path

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ EDIT THIS LINE to match your home directory / preferred location ║
# ╚══════════════════════════════════════════════════════════════════════════╝
GITHUB_URL = 'https://github.com/pes1ug23am910/PROMPTGMF-Bio-Kaggle.git'
PROJECT_DIR = Path('/home/mluser/projects_yash/new_project/PromptGFM-Bio').resolve() # ← change this

# ── Ensure parent directory exists ───────────────────────────────────────
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# ── Clone or pull ─────────────────────────────────────────────────────────
if not (PROJECT_DIR / '.git').is_dir():
 subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, str(PROJECT_DIR)], check=True)
 print(f' Cloned to {PROJECT_DIR}')
else:
 subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull'], check=True)
 print(f' Pulled latest changes')

# ── Set working directory & Python path ──────────────────────────────────
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print(f'Working directory: {os.getcwd()}')

'''

"\nimport os, sys, subprocess\nfrom pathlib import Path\n\n# ╔══════════════════════════════════════════════════════════════════════════╗\n# ║ EDIT THIS LINE to match your home directory / preferred location ║\n# ╚══════════════════════════════════════════════════════════════════════════╝\nGITHUB_URL = 'https://github.com/pes1ug23am910/PROMPTGMF-Bio-Kaggle.git'\nPROJECT_DIR = Path('/home/mluser/projects_yash/new_project/PromptGFM-Bio').resolve() # ← change this\n\n# ── Ensure parent directory exists ───────────────────────────────────────\nPROJECT_DIR.mkdir(parents=True, exist_ok=True)\n\n# ── Clone or pull ─────────────────────────────────────────────────────────\nif not (PROJECT_DIR / '.git').is_dir():\n subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, str(PROJECT_DIR)], check=True)\n print(f' Cloned to {PROJECT_DIR}')\nelse:\n subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull'], check=True)\n print(f' Pulled latest changes')\n\n# ── Set working directory & Python path 

## 6. Create Directory Structure

In [8]:
for d in [DATA_DIR / "raw", DATA_DIR / "processed", DATA_DIR / "splits", CKPT_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("✅ Directories created")

✅ Directories created


## 7. Pre-download BioBERT Model
Downloads ~440 MB of PubMedBERT weights. Only needed once — cache persists on disk.

In [9]:
from huggingface_hub import snapshot_download
from pathlib import Path
import warnings, os

MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

hub_dir = Path(HF_CACHE_DIR) / "hub"
model_cache_name = "models--" + MODEL_NAME.replace("/", "--")
model_snapshots = hub_dir / model_cache_name / "snapshots"

if RESUME_HF_CACHE and model_snapshots.exists() and any(model_snapshots.iterdir()):
    print("⏭️  BioBERT already cached — skipping download")
else:
    print(f"Downloading {MODEL_NAME}")
    print(f"  Cache dir : {HF_CACHE_DIR}")
    print(f"  Size      : ~440 MB")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        snapshot_download(
            repo_id=MODEL_NAME,
            cache_dir=HF_CACHE_DIR,
            ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*", "rust_model*", "*.ot"],
        )
    print("\n✅ BioBERT fully downloaded and cached")

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"]       = "1"
print("✅ Offline mode enabled")

  Cache dir : /home/mluser/projects_yash/new_project/PromptGFM-Bio/hf_cache
  Size      : ~440 MB


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]


✅ BioBERT fully downloaded and cached
✅ Offline mode enabled


## 8. Download Biomedical Datasets
Skipped if `RESUME_DATA=True` and graph exists. Total download: ~1.5 GB.

In [10]:
import subprocess, sys

graph_exists = (DATA_DIR / "processed" / "biomedical_graph.pt").exists()

if RESUME_DATA and graph_exists:
    print("⏭️  Download skipped — data already on disk")
else:
    print("Downloading datasets...")
    script = str(SCRIPTS_DIR / "download_data.py")
    print(f"  Running: {script}")
    result = subprocess.run(
        [sys.executable, script, "--dataset", "all"],
        cwd=str(PROJECT_ROOT),
    )
    print("Download exit code:", result.returncode)
    if result.returncode != 0:
        print("⚠️  Download may have failed — check output above")

  Running: /home/mluser/projects_yash/new_project/PromptGFM-Bio/scripts/download_data.py

PromptGFM-Bio Data Download

Dataset: all
Force re-download: False
Skip failing: True


✓ Successfully downloaded 4/4 datasets

✓ DOWNLOAD COMPLETE

Next steps:
  1. Run preprocessing: python scripts/preprocess_all.py
  2. Check downloaded files in: data/raw/

Download exit code: 0


INFO:src.data.download:======================================================================
INFO:src.data.download:Starting download of all biomedical datasets...
INFO:src.data.download:This may take 30-60 minutes depending on your connection
INFO:src.data.download:Total size: ~1.5 GB
INFO:src.data.download:======================================================================
INFO:src.data.download:
[1/4] BioGRID Protein-Protein Interactions
INFO:src.data.download:BioGRID file already exists at /home/mluser/projects_yash/new_project/PromptGFM-Bio/data/raw/biogrid/BIOGRID-ALL-4.4.224.tab3.zip
INFO:src.data.download:Use force=True to re-download
INFO:src.data.download:
[2/4] STRING Protein Network
INFO:src.data.download:STRING file already exists at /home/mluser/projects_yash/new_project/PromptGFM-Bio/data/raw/string/9606.protein.links.v12.0.txt.gz
INFO:src.data.download:Use force=True to re-download
INFO:src.data.download:
[3/4] DisGeNET Gene-Disease Associations
INFO:src.data.downlo

## 9. Preprocess Data (Build Knowledge Graph)
Skipped if `RESUME_DATA=True` and graph exists.

In [11]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "pandas"], check=True)

CompletedProcess(args=['/home/mluser/micromamba/envs/promptgfm/bin/python', '-m', 'pip', 'install', 'pandas'], returncode=0)

In [ ]:
import subprocess, sys
from pathlib import Path

graph_path = DATA_DIR / "processed" / "biomedical_graph.pt"
script = str(SCRIPTS_DIR / "preprocess_all.py")

if RESUME_DATA and graph_path.exists():
    print(f"⏭️  Preprocessing skipped — graph ready ({graph_path.stat().st_size/1e6:.0f} MB)")
else:
    cmd = [sys.executable, script]

    # If a previous graph exists, force rebuild so STRING path/mapping fixes are applied.
    if graph_path.exists():
        cmd.append("--force")
        print("Existing graph found — forcing rebuild to refresh PPI edges")

    print("Building knowledge graph...")
    print(f"  Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    print("Preprocessing exit code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError("Preprocessing failed — check logs above")

if not graph_path.exists():
    raise RuntimeError("Graph file not created — check logs above")

print(f"✅ Graph ready ({graph_path.stat().st_size/1e6:.0f} MB)")

# Validate that gene-gene edges exist for GNN message passing.
import torch

def _load_graph_cpu(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def _count_ppi_edges(graph_obj):
    edge_types = [
        ("gene", "interacts", "gene"),
        ("gene", "protein_interaction", "gene"),
        ("gene", "ppi", "gene"),
    ]
    for edge_type in edge_types:
        if edge_type in graph_obj.edge_types:
            return int(graph_obj[edge_type].edge_index.shape[1]), edge_type
    return 0, None

graph = _load_graph_cpu(graph_path)
ppi_edge_count, matched_edge_type = _count_ppi_edges(graph)

if ppi_edge_count > 0:
    print(f"✅ Gene-gene edges found: {ppi_edge_count:,} ({matched_edge_type})")
else:
    print("⚠️  No gene-gene edges found after preprocessing.")
    patch_script = PROJECT_ROOT / "add_string_ppi_edges.py"

    if patch_script.exists():
        print(f"  Running fallback patch: {patch_script}")
        patch_result = subprocess.run(
            [sys.executable, str(patch_script)],
            cwd=str(PROJECT_ROOT),
        )
        if patch_result.returncode != 0:
            raise RuntimeError("Fallback PPI patch failed — check logs above")

        graph = _load_graph_cpu(graph_path)
        repaired_count, repaired_edge_type = _count_ppi_edges(graph)
        if repaired_count > 0:
            print(f"✅ Repaired gene-gene edges: {repaired_count:,} ({repaired_edge_type})")
        else:
            raise RuntimeError("Graph still has no gene-gene edges after fallback patch")
    else:
        raise RuntimeError(f"Missing fallback script: {patch_script}")

Building knowledge graph...
  Running: /home/mluser/projects_yash/new_project/PromptGFM-Bio/scripts/preprocess_all.py


/home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/libpyg.so: undefined symbol: _ZN5torch8autograd12VariableInfoC1ERKN2at6TensorE
  import torch_geometric.typing
/home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/torch_scatter/_scatter_cuda.so: undefined symbol: _ZN2at4_ops16div__Tensor_mode4callERNS_6TensorERKS2_St8optionalIN3c1017basic_string_viewIcEEE
  import torch_geometric.typing
/home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Sta


PromptGFM-Bio Enhanced Preprocessing Pipeline

Configuration:
  Force reprocess: False
  HPO Bridge: True
  Orphadata: True
  UniProt: False
  Pathways: False


✓ PREPROCESSING COMPLETE

Next steps:
  1. Create dataset splits: python -m src.data.dataset
  2. Check graph file: data/processed/biomedical_graph.pt
  3. View statistics: data/processed/biomedical_graph_stats.txt

Preprocessing exit code: 0
✅ Graph ready (313 MB)


In [13]:
'''
#use if needed/ previous cell fails
# ─────────────────────────────────────────────────────────────
# 4. Install & Verify All Dependencies (ROBUST VERSION)
# ─────────────────────────────────────────────────────────────

import subprocess, sys, importlib

def pip_install(packages):
    print(f"Installing: {packages}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet"] + packages,
        check=True
    )

def ensure_import(pkg_name, pip_name=None):
    """Ensure a package is installed and importable"""
    pip_name = pip_name or pkg_name
    try:
        importlib.import_module(pkg_name)
        print(f"✅ {pkg_name} already available")
    except ImportError:
        print(f"⚠️  {pkg_name} missing → installing...")
        pip_install([pip_name])
        try:
            importlib.import_module(pkg_name)
            print(f"✅ {pkg_name} installed successfully")
        except ImportError:
            raise RuntimeError(f"❌ Failed to import {pkg_name} even after install")

# ── Upgrade core tooling ──────────────────────────────────────
pip_install(["--upgrade", "pip", "setuptools", "wheel"])

# ── Critical deps (MUST exist for scripts to run) ─────────────
critical_deps = {
    "pandas": "pandas>=2.2.0",        # 🔴 REQUIRED (your failure)
    "numpy": "numpy",
    "yaml": "pyyaml",
}

# ── Project deps ─────────────────────────────────────────────
project_deps = {
    "transformers": "transformers>=4.40.0",
    "sentence_transformers": "sentence-transformers>=2.7.0",
    "Bio": "biopython>=1.84",
    "networkx": "networkx>=3.2",
    "wandb": "wandb>=0.17.0",
    "dotenv": "python-dotenv>=1.0.0",
    "huggingface_hub": "huggingface_hub",
}

# ── Ensure everything is importable ───────────────────────────
print("\n🔍 Checking critical dependencies...")
for module, pip_pkg in critical_deps.items():
    ensure_import(module, pip_pkg)

print("\n🔍 Checking project dependencies...")
for module, pip_pkg in project_deps.items():
    ensure_import(module, pip_pkg)

print("\n✅ All dependencies verified and ready")
'''

'\n#use if needed/ previous cell fails\n# ─────────────────────────────────────────────────────────────\n# 4. Install & Verify All Dependencies (ROBUST VERSION)\n# ─────────────────────────────────────────────────────────────\n\nimport subprocess, sys, importlib\n\ndef pip_install(packages):\n    print(f"Installing: {packages}")\n    subprocess.run(\n        [sys.executable, "-m", "pip", "install", "--quiet"] + packages,\n        check=True\n    )\n\ndef ensure_import(pkg_name, pip_name=None):\n    """Ensure a package is installed and importable"""\n    pip_name = pip_name or pkg_name\n    try:\n        importlib.import_module(pkg_name)\n        print(f"✅ {pkg_name} already available")\n    except ImportError:\n        print(f"⚠️  {pkg_name} missing → installing...")\n        pip_install([pip_name])\n        try:\n            importlib.import_module(pkg_name)\n            print(f"✅ {pkg_name} installed successfully")\n        except ImportError:\n            raise RuntimeError(f"❌ Fail

## 10. W&B Login

In [14]:
import os

WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print("✅ W&B logged in")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("W&B disabled — add WANDB_API_KEY to .env to enable")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/mluser/.netrc
wandb: Currently logged in as: pes1ug23am910 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ W&B logged in


## 11. 🧠 Generate VRAM-Aware Config

Reads `configs/kaggle_config.yaml` and patches it based on free VRAM **right now**.

| Free VRAM | batch_size | grad_accum | effective_batch | workers |
|---|---|---|---|---|
| ≥ 20 GB | 768 | 1 | 768 | 8 |
| 16–20 GB | 512 | 1 | 512 | 6 |
| 12–16 GB | 384 | 1 | 384 | 4 |
| 8–12 GB | 256 | 1 | 256 | 4 |
| 5–8 GB | 128 | 2 | 256 | 2 |
| < 5 GB | 64 | 4 | 256 | 2 |

In [15]:
import subprocess, os, yaml
import torch

# ── 1. Probe free VRAM ───────────────────────────────────────────────────
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total,memory.free",
     "--format=csv,nounits,noheader"],
    capture_output=True, text=True
)
smi_used, smi_total, smi_free = [int(x.strip()) for x in result.stdout.strip().split(",")]
usable_mb = smi_free - 2048

print(f"GPU VRAM: {smi_total:,} MiB total · {smi_used:,} MiB used · {smi_free:,} MiB free")
print(f"Usable for training: {usable_mb:,} MiB (after 2 GB safety margin)")
print()

# ── 2. Pick batch_size + gradient accumulation ───────────────────────────
if usable_mb >= 20000:
    batch_size, grad_accum, num_workers = 768, 1, 8
elif usable_mb >= 16000:
    batch_size, grad_accum, num_workers = 512, 1, 6
elif usable_mb >= 12000:
    batch_size, grad_accum, num_workers = 384, 1, 4
elif usable_mb >= 8000:
    batch_size, grad_accum, num_workers = 256, 1, 4
elif usable_mb >= 5000:
    batch_size, grad_accum, num_workers = 128, 2, 2
else:
    batch_size, grad_accum, num_workers = 64, 4, 2

effective_batch = batch_size * grad_accum

print(f"╔══════════════════════════════════════════════╗")
print(f"║  batch_size       = {batch_size:<6}                   ║")
print(f"║  grad_accum       = {grad_accum:<6}                   ║")
print(f"║  effective_batch   = {effective_batch:<6}                  ║")
print(f"║  num_workers      = {num_workers:<6}                   ║")
print(f"╚══════════════════════════════════════════════╝")
print()

# ── 3. Read base config and patch ────────────────────────────────────────
base_cfg_path = CONFIGS_DIR / "kaggle_config.yaml"
ws_cfg_path   = CONFIGS_DIR / "workstation_config.yaml"

if base_cfg_path.exists():
    with open(base_cfg_path) as f:
        cfg = yaml.safe_load(f)

    def patch_dict(d, patches):
        if not isinstance(d, dict):
            return
        for k, v in patches.items():
            if k in d:
                print(f"  patching {k}: {d[k]} → {v}")
                d[k] = v
        for child in d.values():
            if isinstance(child, dict):
                patch_dict(child, patches)

    patches = {
        "batch_size":                   batch_size,
        "num_workers":                  num_workers,
        "gradient_accumulation_steps":  grad_accum,
        "pin_memory":                   True,
        "persistent_workers":           num_workers > 0,
    }

    print("Patching config:")
    patch_dict(cfg, patches)

    if "gradient_accumulation_steps" not in str(cfg):
        target = cfg.get("training", cfg)
        target["gradient_accumulation_steps"] = grad_accum
        print(f"  added gradient_accumulation_steps: {grad_accum} (new key)")

    with open(ws_cfg_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

    print(f"\n✅ Wrote {ws_cfg_path}")
else:
    print(f"⚠️  {base_cfg_path} not found")

# ── 4. RTX 4090 performance flags ────────────────────────────────────────
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True
print("✅ TF32 matmul ✓ · TF32 cuDNN ✓ · cuDNN benchmark ✓")

GPU VRAM: 24,564 MiB total · 120 MiB used · 23,953 MiB free
Usable for training: 21,905 MiB (after 2 GB safety margin)

╔══════════════════════════════════════════════╗
║  batch_size       = 768                      ║
║  grad_accum       = 1                        ║
║  effective_batch   = 768                     ║
║  num_workers      = 8                        ║
╚══════════════════════════════════════════════╝

Patching config:
  patching batch_size: 256 → 768
  patching num_workers: 4 → 8
  patching pin_memory: True → True
  added gradient_accumulation_steps: 1 (new key)

✅ Wrote /home/mluser/projects_yash/new_project/PromptGFM-Bio/configs/workstation_config.yaml
✅ TF32 matmul ✓ · TF32 cuDNN ✓ · cuDNN benchmark ✓


## 12. Train

Uses `configs/workstation_config.yaml` auto-generated above with VRAM-aware batch_size.

In [16]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"]       = "1"
print("✅ Offline mode confirmed")

✅ Offline mode confirmed


In [17]:
import torch, subprocess, sys

# ── Config ────────────────────────────────────────────────────────────────
ws_cfg = CONFIGS_DIR / "workstation_config.yaml"
config = str(ws_cfg) if ws_cfg.exists() else str(CONFIGS_DIR / "kaggle_config.yaml")
print(f"Using config: {config}")

# ── Auto-detect checkpoints for resume ────────────────────────────────────
ckpts = sorted(
    CKPT_DIR.glob("checkpoint_epoch_*.pt"),
    key=lambda p: int(p.stem.split("_")[-1])
) if CKPT_DIR.exists() else []

RESUME = RESUME_CHECKPOINTS or bool(ckpts)

base_args = [str(SCRIPTS_DIR / "train.py"), "--config", config]
if RESUME and ckpts:
    last_ckpt = str(ckpts[-1])
    base_args += ["--resume-checkpoint", last_ckpt]
    print(f"Resuming from: {last_ckpt}")
elif RESUME:
    print("RESUME_CHECKPOINTS=True but no checkpoints found — starting fresh")
    RESUME = False

# ── Launch ────────────────────────────────────────────────────────────────
n_gpus = torch.cuda.device_count()
print(f"GPUs available: {n_gpus}")

if n_gpus > 1:
    import shutil
    torchrun = shutil.which("torchrun") or "torchrun"
    cmd = [torchrun, f"--nproc_per_node={n_gpus}", "--master_port=29500"] + base_args
    print(f"Launching DDP on {n_gpus} GPUs")
else:
    cmd = [sys.executable] + base_args
    print("Single-GPU launch (RTX 4090)")

print("Running:", " ".join(cmd))
print()
result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
print("\nTraining exit code:", result.returncode)

Using config: /home/mluser/projects_yash/new_project/PromptGFM-Bio/configs/workstation_config.yaml
GPUs available: 1
Single-GPU launch (RTX 4090)
Running: /home/mluser/micromamba/envs/promptgfm/bin/python /home/mluser/projects_yash/new_project/PromptGFM-Bio/scripts/train.py --config /home/mluser/projects_yash/new_project/PromptGFM-Bio/configs/workstation_config.yaml



/home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/libpyg.so: undefined symbol: _ZN5torch8autograd12VariableInfoC1ERKN2at6TensorE
  import torch_geometric.typing
/home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/torch_scatter/_scatter_cuda.so: undefined symbol: _ZN2at4_ops16div__Tensor_mode4callERNS_6TensorERKS2_St8optionalIN3c1017basic_string_viewIcEEE
  import torch_geometric.typing
/home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Sta


Training exit code: 0


## 12.5. 📊 Post-Training GPU Check
See peak memory usage — helps decide if batch_size can go higher next time.

In [18]:
import subprocess, torch

subprocess.run(["nvidia-smi"])

if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated(0) / 1e9
    reserved = torch.cuda.max_memory_reserved(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print()
    print(f"PyTorch peak allocated : {peak:.1f} GB")
    print(f"PyTorch peak reserved  : {reserved:.1f} GB")
    print(f"Total VRAM             : {total:.1f} GB")
    print(f"Headroom               : {total - reserved:.1f} GB")
    if total - reserved > 4.0:
        print("💡 You have headroom — try a larger batch_size next run.")
    elif total - reserved < 1.0:
        print("⚠️  Very tight — reduce batch_size if you see OOM errors.")

Thu Apr  2 17:14:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.65.06              Driver Version: 580.65.06      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:01:00.0 Off |                  Off |
| 34%   55C    P2             66W /  450W |     120MiB /  24564MiB |     24%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 13. 💾 Backup to GitHub / HuggingFace

> **CRITICAL**: Workstation wipes all data after 5 days!

In [19]:
import subprocess, os

os.environ["GIT_TERMINAL_PROMPT"] = "0"

if not (PROJECT_ROOT / ".git").is_dir():
    print("⚠️  Not a git repo — cannot push. Back up manually:")
    print(f"   scp -r {CKPT_DIR} your-laptop:~/backups/")
else:
    for pattern in ["checkpoints/", "logs/"]:
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "add", "-f", pattern])

    status = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain"],
        capture_output=True, text=True
    )

    if status.stdout.strip():
        ckpts = sorted(
            CKPT_DIR.glob("checkpoint_epoch_*.pt"),
            key=lambda f: int(f.stem.split("_")[-1])
        ) if CKPT_DIR.exists() else []
        epoch = ckpts[-1].stem.split("_")[-1] if ckpts else "?"

        subprocess.run([
            "git", "-C", str(PROJECT_ROOT), "commit", "-m",
            f"Workstation training: epoch {epoch} checkpoints"
        ])
        result = subprocess.run(["git", "-C", str(PROJECT_ROOT), "push"])
        if result.returncode == 0:
            print("✅ Pushed to GitHub")
        else:
            print("❌ Git push failed — check GITHUB_TOKEN in .env")
    else:
        print("Nothing new to commit")

print()
print("For large files, consider HuggingFace Hub:")
print(f"  huggingface-cli upload your-username/promptgfm-bio {HF_CACHE_DIR} --repo-type model")

⚠️  Not a git repo — cannot push. Back up manually:
   scp -r /home/mluser/projects_yash/new_project/PromptGFM-Bio/checkpoints/promptgfm_film your-laptop:~/backups/

For large files, consider HuggingFace Hub:
  huggingface-cli upload your-username/promptgfm-bio /home/mluser/projects_yash/new_project/PromptGFM-Bio/hf_cache --repo-type model


## 14. Quick Evaluation

In [20]:
'''
import subprocess, sys

ws_cfg = CONFIGS_DIR / "workstation_config.yaml"
config = str(ws_cfg) if ws_cfg.exists() else str(CONFIGS_DIR / "kaggle_config.yaml")

best = CKPT_DIR / "best_model.pt"
if not best.exists():
    print("No best_model.pt yet — run more training epochs first")
else:
    result = subprocess.run(
        [sys.executable, str(SCRIPTS_DIR / "evaluate.py"),
         "--config", config,
         "--checkpoint", str(best)],
        cwd=str(PROJECT_ROOT),
    )
    print("Evaluation exit code:", result.returncode)
'''

'\nimport subprocess, sys\n\nws_cfg = CONFIGS_DIR / "workstation_config.yaml"\nconfig = str(ws_cfg) if ws_cfg.exists() else str(CONFIGS_DIR / "kaggle_config.yaml")\n\nbest = CKPT_DIR / "best_model.pt"\nif not best.exists():\n    print("No best_model.pt yet — run more training epochs first")\nelse:\n    result = subprocess.run(\n        [sys.executable, str(SCRIPTS_DIR / "evaluate.py"),\n         "--config", config,\n         "--checkpoint", str(best)],\n        cwd=str(PROJECT_ROOT),\n    )\n    print("Evaluation exit code:", result.returncode)\n'

In [21]:
import subprocess, sys

# 🔴 FORCE correct config (no auto-switching)
config = str(CONFIGS_DIR / "workstation_config.yaml")

best = CKPT_DIR / "best_model.pt"

if not best.exists():
    print("No best_model.pt yet — run more training epochs first")
else:
    result = subprocess.run(
        [
            sys.executable,
            str(SCRIPTS_DIR / "evaluate.py"),
            "--config", config,
            "--checkpoint", str(best),
        ],
        cwd=str(PROJECT_ROOT),
    )
    print("Evaluation exit code:", result.returncode)

/home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/libpyg.so: undefined symbol: _ZN5torch8autograd12VariableInfoC1ERKN2at6TensorE
  import torch_geometric.typing
/home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/torch_scatter/_scatter_cuda.so: undefined symbol: _ZN2at4_ops16div__Tensor_mode4callERNS_6TensorERKS2_St8optionalIN3c1017basic_string_viewIcEEE
  import torch_geometric.typing
/home/mluser/micromamba/envs/promptgfm/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Sta

Evaluation exit code: 0


## 15. Disk Usage Check

In [22]:
import subprocess
from pathlib import Path

subprocess.run(["df", "-h", str(PROJECT_ROOT)])
print()

for label, path in [("hf_cache", HF_CACHE_DIR), ("data", DATA_DIR),
                     ("checkpoints", CKPT_DIR), ("logs", LOGS_DIR)]:
    p = Path(path)
    if p.exists():
        result = subprocess.run(["du", "-sh", str(p)], capture_output=True, text=True)
        print(result.stdout.strip())

Filesystem      Size  Used Avail Use% Mounted on
overlay         492G  171G  296G  37% /

421M	/home/mluser/projects_yash/new_project/PromptGFM-Bio/hf_cache
3.7G	/home/mluser/projects_yash/new_project/PromptGFM-Bio/data
7.1G	/home/mluser/projects_yash/new_project/PromptGFM-Bio/checkpoints/promptgfm_film
4.0K	/home/mluser/projects_yash/new_project/PromptGFM-Bio/logs


## 16. 🔧 Manual Override (Optional)

If auto-detected batch_size isn't right, uncomment and edit below, then re-run Step 12.

In [23]:
# import yaml
#
# MANUAL_BATCH_SIZE  = 384
# MANUAL_GRAD_ACCUM  = 1
# MANUAL_NUM_WORKERS = 4
#
# ws_cfg_path = PROJECT_ROOT / "configs" / "workstation_config.yaml"
# if ws_cfg_path.exists():
#     with open(ws_cfg_path) as f:
#         cfg = yaml.safe_load(f)
#
#     def patch(d, k, v):
#         if isinstance(d, dict):
#             if k in d: d[k] = v
#             for child in d.values(): patch(child, k, v)
#
#     patch(cfg, "batch_size", MANUAL_BATCH_SIZE)
#     patch(cfg, "gradient_accumulation_steps", MANUAL_GRAD_ACCUM)
#     patch(cfg, "num_workers", MANUAL_NUM_WORKERS)
#
#     with open(ws_cfg_path, "w") as f:
#         yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
#     print(f"✅ Updated: batch={MANUAL_BATCH_SIZE}, accum={MANUAL_GRAD_ACCUM}")